# Pipeline walkthrough

This notebook walks through the pipeline in five parts: data checks, validation set-up, modelling, metric choice and
results. It uses the same problem definition, features, folds and metrics as `feature_and_model_exploration.ipynb`, and then adds the
parts that only exist in the package: the split over DCs, P80 and P90 with coverage and pinball loss, the backtest without a
known schedule, the feature importance plot, and the peak-day and holiday-week results. Those parts are marked as such. **Code cells** run the package in `src/cs_outbound_coli_prediction`, and check it against the numbers of the experiment notebook.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

from cs_outbound_coli_prediction import config
from cs_outbound_coli_prediction.backtest import bias, usable_rows, wape
from cs_outbound_coli_prediction.cli import score_models
from cs_outbound_coli_prediction.data import load_deliveries
from cs_outbound_coli_prediction.features import build_features, in_christmas_period
from cs_outbound_coli_prediction.models import FallbackForecaster, Forecaster
from cs_outbound_coli_prediction.quantiles import pinball_loss

deliveries = load_deliveries(config.input_path)

# values copied from the saved outputs of feature_and_model_exploration.ipynb
EXPERIMENT_FOLD_SIZES = [(21928, 9600), (31528, 9600), (41128, 9600), (50728, 9600)]
SELECTED_FEATURES = ['qty_last_1', 'qty_last_2', 'qty_last_3', 'qty_lag_52', 'mean_4', 'std_4', 'mean_13', 'std_13', 'mean_52', 'std_52', 'week_of_year', 'month', 'week_sin', 'week_cos', 'xmas_week', 'xmas_1w_before', 'xmas_2w_before', 'n_days', 'days_ratio', 'horizon', 'country_store', 'store', 'sales_last_1', 'sales_last_2', 'sales_last_3', 'sales_mean_4']
BEST_LGB = {'n_estimators': 400, 'learning_rate': 0.03, 'num_leaves': 31, 'min_child_samples': 40, 'subsample': 0.7, 'subsample_freq': 1, 'colsample_bytree': 0.7, 'reg_lambda': 1.0, 'objective': 'l1'}
BEST_XGB = {'n_estimators': 200, 'learning_rate': 0.03, 'max_depth': 6, 'min_child_weight': 50, 'subsample': 0.7, 'colsample_bytree': 0.7, 'reg_lambda': 20.0, 'tree_method': 'hist', 'enable_categorical': True, 'objective': 'count:poisson'}
EXPERIMENT_HOLDOUT = {"model": 0.113975, "model_bias": -0.000545,
                      "fallback": 0.124846, "fallback_bias": 0.006935}

## 1. Data checks

In [ ]:
print(f"{len(deliveries):,} rows, {deliveries['store'].nunique()} stores, "
      f"weeks {deliveries['Monday_of_week'].min().date()} to {deliveries['Monday_of_week'].max().date()}")
print("missing values per column:", deliveries.isna().sum()[lambda missing: missing > 0].to_dict() or "none")

# stores with an irregular history: they are the ones the zero-week rule and the history rule apply to
first = deliveries.groupby("store")["Monday_of_week"].min()
last = deliveries.groupby("store")["Monday_of_week"].max()
weeks_in_span = (last - first).dt.days // 7 + 1

weeks_with_deliveries = deliveries.groupby("store")["Monday_of_week"].nunique()

print("stores starting after the first week:", {s: d.date().isoformat() for s, d in first[first > first.min()].items()})
print("weeks without any delivery inside a store's own history:", (weeks_in_span - weeks_with_deliveries)[lambda gap: gap > 0].to_dict())

In [ ]:
# volume around Christmas: the Christmas week (the ISO week with 25 December) and the weeks after it, as the package defines them
weekly_total = deliveries.groupby("Monday_of_week")["delivery_quantity"].sum()
holiday = in_christmas_period(weekly_total.index.to_series())

print(f"typical week (all other weeks): median {weekly_total[~holiday].median():,.0f} colli, "
      f"range {weekly_total[~holiday].min():,.0f} to {weekly_total[~holiday].max():,.0f}")

print(f"Christmas week and the {config.christmas_weeks_after} after it:")

print(weekly_total[holiday].round(0).astype(int).rename("colli").to_frame().assign(
    share_of_typical=lambda t: (t["colli"] / weekly_total[~holiday].median()).round(2)).to_string())

In [ ]:
# the leakage test: change everything after an origin, the features of the rows forecast from it must not move
features = build_features(deliveries)
weeks = features["Monday_of_week"].drop_duplicates().sort_values()
origin = weeks.iloc[70]
changed = deliveries.copy()
changed.loc[changed["Monday_of_week"] > origin, ["delivery_quantity", "sales_index"]] *= 1000

key = ["store", "Monday_of_week", "horizon"]

both = features.merge(build_features(changed), on=key, suffixes=("_a", "_b"))
history_columns = [c for c in config.feature_columns if c.startswith(("qty_", "mean_", "std_", "sales_"))]

for horizon in range(1, config.max_horizon + 1):
    known = both[(both["horizon"] == horizon) & (both["Monday_of_week"] <= origin + pd.Timedelta(weeks=horizon))]
    for column in history_columns:
        pd.testing.assert_series_equal(known[f"{column}_a"], known[f"{column}_b"], check_names=False)
print("no-leakage checks passed")

## 2. Validation set-up

In [ ]:
rows = usable_rows(features)
all_weeks = rows["Monday_of_week"].drop_duplicates().sort_values().reset_index(drop=True)
dev_weeks, holdout_weeks = all_weeks.iloc[:-config.holdout_weeks], all_weeks.iloc[-config.holdout_weeks:]
dev, holdout = rows[rows["Monday_of_week"].isin(dev_weeks)], rows[rows["Monday_of_week"].isin(holdout_weeks)]
train_for_holdout = dev[dev["Monday_of_week"].isin(dev_weeks.iloc[:-(config.max_horizon - 1)])]

folds = [(dev[dev["Monday_of_week"].isin(dev_weeks.iloc[train])], dev[dev["Monday_of_week"].isin(dev_weeks.iloc[test])])
         for train, test in TimeSeriesSplit(n_splits=4, test_size=12, gap=config.max_horizon - 1).split(dev_weeks)]
sizes = [(len(train), len(test)) for train, test in folds]
print("fold sizes (train, test):", sizes)
print("same as feature_and_model_exploration.ipynb:", sizes == EXPERIMENT_FOLD_SIZES)
print(f"holdout {holdout_weeks.iloc[0].date()} to {holdout_weeks.iloc[-1].date()}, "
      f"training for it stops {train_for_holdout['Monday_of_week'].max().date()} ({len(train_for_holdout):,} rows)")

weekly_total = deliveries.groupby("Monday_of_week")["delivery_quantity"].sum() / 1000
fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(weekly_total.index, weekly_total.values, color="#1F4E79")
for number, (train, test) in enumerate(folds):
    ax.axvspan(test["Monday_of_week"].min(), test["Monday_of_week"].max(), color="#1F4E79", alpha=0.12 + 0.04 * number,
               label="CV test windows" if number == 0 else None)
ax.axvspan(holdout_weeks.iloc[0], holdout_weeks.iloc[-1], color="#D9822B", alpha=0.3, label="holdout")
ax.set_ylabel("colli per week (thousands)")
ax.set_title("Four cross-validation test windows and the holdout")
ax.legend(loc="lower left")
plt.show()

## 3. Modelling

The modelling steps of the experiment notebook, in order, with its saved results. Every choice was made on the
cross-validation folds, never on the holdout.

**Step 4, baselines and default models.** Four simple baselines (the last known week, the same week last year, the trailing
13-week and 52-week mean) against default LightGBM and XGBoost, then the same models with an absolute-error (L1) and a
Poisson objective. The best objective per model by mean CV WAPE was used from there on: **L1 for LightGBM, Poisson for XGBoost.**

**Cross-validated WAPE of baselines and default models (4 folds)**

```text
                         WAPE mean  WAPE std  WAPE worst fold  bias mean
LightGBM (l1)             0.137450  0.019488         0.153884   0.015344
avg(LGB, XGB) (l1)        0.140279  0.023404         0.162231   0.021087
avg(LGB, XGB) (poisson)   0.141059  0.024539         0.167305   0.021717
LightGBM (poisson)        0.141311  0.024516         0.168468   0.021555
XGBoost (poisson)         0.142294  0.025133         0.168586   0.021879
LightGBM (default)        0.145222  0.027988         0.174737   0.024531
XGBoost (l1)              0.145525  0.027650         0.173060   0.026831
avg(LGB, XGB) (default)   0.146614  0.028114         0.174968   0.027726
trailing mean_52          0.149599  0.022068         0.168829   0.024386
XGBoost (default)         0.150604  0.028598         0.178624   0.030920
trailing mean_13          0.157758  0.041379         0.210513   0.017278
seasonal: lag_52          0.171131  0.027220         0.207273   0.027394
naive: last known week    0.190447  0.045937         0.257112   0.008329
```

**Step 5, features.** Ablation (CV with each feature group removed) and permutation importance. A group is only dropped if
removing it improves CV WAPE by at least 0.001. None was, so all 26 features stay.

**Ablation: WAPE with each feature group removed**

```text
                     WAPE mean  bias mean    fold 0    fold 1    fold 2  \
all features          0.137450   0.015344  0.153884  0.127423  0.153587   
without lags_short    0.137446   0.015539  0.154408  0.127621  0.153468   
without lags_long     0.137193   0.014964  0.153884  0.127423  0.151606   
without rolling       0.138671   0.021214  0.152488  0.129278  0.156266   
without calendar      0.140384   0.009005  0.141885  0.158916  0.141506   
without schedule      0.143823   0.033506  0.166710  0.126261  0.162548   
without horizon       0.138309   0.016461  0.158049  0.128106  0.152973   
without categorical   0.139618   0.014382  0.151370  0.131048  0.159285   
without sales         0.137578   0.015694  0.156565  0.126720  0.151516   

                       fold 3  change vs all  
all features         0.114904       0.000000  
without lags_short   0.114285      -0.000004  
without lags_long    0.115857      -0.000257  
without rolling      0.116654       0.001222  
without calendar     0.119230       0.002935  
without schedule     0.119771       0.006373  
without horizon      0.114106       0.000859  
without categorical  0.116770       0.002168  
without sales        0.115510       0.000128
```

**Permutation importance (increase in MAE when a feature is shuffled)**

```text
                mae_increase  std_over_folds
store             272.479657       20.078888
mean_13           254.431037       44.427301
n_days             65.050446       28.895924
week_of_year       18.305721       30.063638
mean_4              5.674546        5.050798
std_52              0.832149        1.166110
week_sin            0.782924        2.837764
days_ratio          0.556218        2.415961
qty_lag_52          0.282096        2.544831
month               0.251770        0.194423
sales_last_2        0.160463        0.343541
qty_last_3          0.039385        0.048788
xmas_2w_before      0.018472        0.036943
xmas_1w_before      0.016529        0.033059
horizon             0.015210        0.007097
xmas_week          -0.025817        0.051634
sales_last_3       -0.063465        0.058207
qty_last_1         -0.069542        0.061040
qty_last_2         -0.087944        0.153935
std_4              -0.091926        0.108896
country_store      -0.196819        0.640308
mean_52            -0.280905        1.001060
std_13             -0.327329        0.578315
sales_last_1       -1.003845        1.609031
sales_mean_4       -1.685694        3.099841
week_cos           -4.573418       11.076262
```

**Step 6, training size.** All history against the last 52, 35 and 26 weeks. A shorter window is only used if clearly better.

**WAPE by training window**

```text
             WAPE mean  WAPE std  train rows (avg)    fold 0    fold 1  \
all history   0.137450  0.019488          36328.00  0.153884  0.127423   
52 weeks      0.137636  0.018761          33897.25  0.153884  0.127423   
35 weeks      0.150944  0.021158          26473.00  0.153884  0.157115   
26 weeks      0.187881  0.048075          20710.75  0.233241  0.171921   

               fold 2    fold 3  
all history  0.153587  0.114904  
52 weeks     0.152913  0.116322  
35 weeks     0.171436  0.121340  
26 weeks     0.218947  0.127415
```

**Step 7, tuning.** A random search of 20 trials per model on the selected features, scored by mean CV WAPE. The best trial of
each is what the package uses, as the check below shows.

In [ ]:
EXPERIMENT_FEATURES = SELECTED_FEATURES
print("package features equal the selected features of the experiment (same set, order may differ):",
      sorted(config.feature_columns) == sorted(EXPERIMENT_FEATURES))
print("LightGBM:", {k: v for k, v in config.lgb_params.items() if k not in ("random_state", "n_jobs", "verbose")})
print("XGBoost :", {k: v for k, v in config.xgb_params.items() if k not in ("random_state", "n_jobs")})
print("same as the best trials of the experiment:", (
    all(config.lgb_params[k] == v for k, v in BEST_LGB.items()) and all(config.xgb_params[k] == v for k, v in BEST_XGB.items())))

**Step 8, final model.** The average of the tuned LightGBM and XGBoost had the lowest mean CV WAPE and is the model of the
package, `Forecaster`. The best baseline, the trailing 52-week mean, is the package's fallback model.

**After the weekly forecast (from here on only in the package, not in `feature_and_model_exploration.ipynb`).**

1. **Split over delivery dates.** The weekly quantity of a store times the store's average weekday share over the last 26 weeks,
   over the scheduled dates. The experiment scored this split once: see the last table of the results.
2. **Split over DCs.** The store's quantity of a date is split over the DCs scheduled that date, by the history of each DC for
   that store (that weekday if every DC has history on it, otherwise the total).
3. **Sum per DC and date.** This is the staffing level.
4. **P80 and P90.** The point forecast of a DC and date times a factor per DC: the 80% and 90% quantile of actual / forecast over
   past DC-days, from three rolling holdouts of 12 weeks. A DC with fewer than 30 DC-days uses the pooled factors. Quantiles of a
   sum are not the sum of store quantiles, so the factors are fitted on the DC totals themselves.

In [ ]:
# feature importance of the LightGBM of the chosen model, by gain (the experiment used ablation and permutation importance)
model = Forecaster().fit(train_for_holdout)
gain = pd.Series(model.models[0].booster_.feature_importance("gain"), index=config.feature_columns).sort_values()
gain.plot.barh(figsize=(7, 6), color="#1F4E79", title="LightGBM feature importance (gain)")
plt.show()
print((gain / gain.sum()).sort_values(ascending=False).head(6).round(3).to_string())

## 4. Metric choice

Two metrics, as defined in the experiment notebook:

- **WAPE** = sum of absolute errors / sum of actuals. The metric the models were chosen by. It is a share of the total
  quantity, so a store counts in proportion to its volume. It is an absolute-error metric, which is why the absolute-error (L1)
  objective was tried.
- **Bias** = (sum of forecasts - sum of actuals) / sum of actuals. Positive means over-forecast. It shows the direction that
  WAPE hides.

Both are reported for the pooled holdout and per country, volume group, history length, horizon and store.

For the DC and date forecast and its P80 and P90 (package only), two more:

- **Coverage** of P80 and P90: the share of DC-days where the actual is at or below the quantile. It should be 80% and 90%.
- **Pinball loss** at a quantile *q*: an under-forecast costs *q* per colli and an over-forecast *1 - q*, divided by the total
  actual. It scores the whole range, and it is comparable with the pinball loss of the point forecast.

The DC and date forecast is scored by WAPE and bias at three levels: store and week, store and date, and DC and date.

In [ ]:
actual = pd.Series([100.0, 200.0, 300.0])
forecast = pd.Series([90.0, 230.0, 270.0])
print(f"WAPE {wape(actual, forecast):.3f} = (10 + 30 + 30) / 600")
print(f"bias {bias(actual, forecast):+.3f} = (590 - 600) / 600")
print(f"pinball at 0.9: an actual of 100 and a forecast of 80 costs {pinball_loss(pd.Series([100.0]), pd.Series([80.0]), 0.9):.2f}, "
      f"a forecast of 120 costs {pinball_loss(pd.Series([100.0]), pd.Series([120.0]), 0.9):.2f}")

## 5. Results

In [ ]:
# the package on the same holdout: the chosen model and the fallback
for name, predicted in {"Forecaster (package)": model.predict(holdout), "FallbackForecaster (package)": FallbackForecaster().predict(holdout)}.items():
    print(f"{name:30s} holdout WAPE {wape(holdout[config.target], predicted):.4f}  bias {bias(holdout[config.target], predicted):+.4f}")
print(f"{'avg(LGB, XGB) (tuned)':30s} holdout WAPE {EXPERIMENT_HOLDOUT['model']:.4f}  bias {EXPERIMENT_HOLDOUT['model_bias']:+.4f}")
print(f"{'trailing mean_52':30s} holdout WAPE {EXPERIMENT_HOLDOUT['fallback']:.4f}  bias {EXPERIMENT_HOLDOUT['fallback_bias']:+.4f}")

### The DC and date forecast, P80 and P90 (package only)

`score_models` is what the `backtest` and `train` commands run: the holdout scores, three rolling holdouts for the quantiles
and the backtest without a known schedule. It takes a minute or two.

In [ ]:
metrics, tables, calibration = score_models(deliveries)

In [ ]:
# WAPE at each level. "schedule = last week repeated": every holdout week is forecast like a production run without a schedule
# file, from data up to 8 weeks before it, with the delivery lines of the last known week repeated.
levels = {
    "store and week, all horizons": ("holdout_wape_main", "holdout_wape_fallback"),
    "store and week, 8 weeks ahead": ("holdout_wape_main_horizon_8", None),
    "store and delivery date": ("holdout_wape_store_date", None),
    "DC and delivery date, known schedule": ("holdout_wape_dc_date", None),
    "DC and delivery date, schedule = last week repeated": ("holdout_wape_dc_date_repeated_schedule_main", "holdout_wape_dc_date_repeated_schedule_fallback"),
}
results = pd.DataFrame({name: {"model": metrics[main], "fallback": metrics.get(fallback, np.nan)} for name, (main, fallback) in levels.items()}).T
results["model vs fallback"] = results["model"] / results["fallback"] - 1
results.style.format({"model": "{:.3f}", "fallback": "{:.3f}", "model vs fallback": "{:+.1%}"}, na_rep="-")

In [ ]:
print({name: round(value, 3) for name, value in metrics.items() if "coverage" in name or "pinball" in name})
dc = tables["dc"].sort_index()
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(dc.index, dc["p80_coverage"], color="#1F4E79", label="P80 coverage")
ax.axhline(0.8, color="#D9822B", linestyle="--", label="target 80%")
ax.set(title="P80 coverage per DC (newest holdout)", ylim=(0, 1))
ax.tick_params(axis="x", rotation=90)
ax.legend()
plt.show()
print(f"P80 coverage per DC: lowest {dc['p80_coverage'].idxmin()} {dc['p80_coverage'].min():.0%}, highest {dc['p80_coverage'].idxmax()} {dc['p80_coverage'].max():.0%}")

In [ ]:
# peak days (the busiest 10% of DC-days of each DC) and holiday weeks (Christmas week + 4), all three rolling holdouts
tables["day_type"][["dc_days", "main_wape", "fallback_wape", "main_bias", "main_p80_coverage", "main_p90_coverage"]]

**Results**

- **The model beats the best baseline.** Cross-validated WAPE is 0.134 against 0.150 for the trailing 52-week mean (10.6%
  lower), and the worst fold is 0.150 against 0.169. On the holdout it is 0.114 against 0.125 (8.7% lower), with a bias of -0.001.
- **The holdout is the easier end.** The folds of the cross-validation vary more than that (fold 0 to 3 for LightGBM with L1 and default
  parameters: 0.154, 0.127, 0.154, 0.115), so the holdout WAPE is not the whole picture.
- **A longer horizon costs little.** On the holdout WAPE goes from 0.111 (1 week ahead) to 0.116 (8 weeks ahead).
  The bias moves from about 0 to -0.004.
- **The schedule matters most.** In the ablation, removing the delivery days (`schedule`) raises CV WAPE by 0.0064, the largest
  change, then the calendar (0.0029), the categorical features (0.0022), the rolling statistics (0.0012) and the horizon
  (0.0009). The sales features add 0.0001. In the permutation importance `store`, `mean_13` and `n_days` come first.
- **The last 52 weeks are as good as all history, shorter is worse** (0.1376 against 0.1375, then 0.151 and 0.188), so all
  history is used.
- **The weekday split adds little.** Splitting the actual weekly totals gives 0.160 per store and delivery date, the
  end-to-end forecast gives 0.197, and an equal split gives 0.199.
- **Weak spots seen in the experiment.** Stores without a 52-week lag are worse than the trailing mean in the
  cross-validation (0.178 against 0.140, bias +13%, default LightGBM). The worst store on the holdout is S0084 (0.297), and six
  stores are above 0.2. Two of them, S0024 and S0073, have a large positive bias (+0.22 and +0.13).